# Proximal Policy Optimization (PPO) on Pendulum-v1

This notebook trains a PPO agent on the Pendulum environment with an action normalizer wrapper. It also demonstrates saving, loading from disk, and publishing/loading models via Hugging Face Hub.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import numpy as np
from tensoraerospace.agent import PPO


env = gym.make('Pendulum-v1')

class ActionNormalizer(gym.ActionWrapper):
    """Rescale and relocate the actions."""

    def action(self, action: np.ndarray) -> np.ndarray:
        """Change the range (-1, 1) to (low, high)."""
        low = self.action_space.low
        high = self.action_space.high

        scale_factor = (high - low) / 2
        reloc_factor = high - scale_factor

        action = action * scale_factor + reloc_factor
        action = np.clip(action, low, high)

        return action

    def reverse_action(self, action: np.ndarray) -> np.ndarray:
        """Change the range (low, high) to (-1, 1)."""
        low = self.action_space.low
        high = self.action_space.high

        scale_factor = (high - low) / 2
        reloc_factor = high - scale_factor

        action = (action - reloc_factor) / scale_factor
        action = np.clip(action, -1.0, 1.0)


        return action

In [2]:

env = ActionNormalizer(env)
env.reset()

agent = PPO(env, gamma=0.9, max_episodes = 100)

agent.train()

100%|██████████| 100/100 [02:57<00:00,  1.77s/it]


## Train the PPO Agent

Wrap the environment with `ActionNormalizer` to rescale actions to [-1, 1], then create and train the PPO agent.

In [ ]:
SAVED_DIR = agent.save()
print('saved to:', SAVED_DIR)


## Save the Trained Model

In [ ]:
import os
_HF_TOKEN = os.environ.get('HF_TOKEN', '')
if _HF_TOKEN:
    agent.publish_to_hub('Mr8bit/ppo-pendelium', str(SAVED_DIR), access_token=_HF_TOKEN)
else:
    print('HF_TOKEN not set, skipping publish_to_hub demo')


## Publish to Hugging Face Hub

In [ ]:
from tensoraerospace.agent import PPO

agent = PPO.from_pretrained(str(SAVED_DIR))


## Load Agent from Local Checkpoint

In [ ]:
import os
_HF_TOKEN = os.environ.get('HF_TOKEN', '')
if _HF_TOKEN:
    from tensoraerospace.agent import PPO
    agent = PPO.from_pretrained(repo_name='Mr8bit/ppo-pendelium', access_token=_HF_TOKEN)
else:
    print('HF_TOKEN not set, skipping from_pretrained hub demo')


## Load Agent from Hugging Face Hub